# Lab 00 · Arrume a casa antes de modelar
**Machine Learning na Prática · Dia 1 · Módulo 01: Dados & Datasets**

Neste notebook você vai fazer o que todo projeto de ML faz **antes** de treinar qualquer modelo.
Aqui **não há modelo nem métricas**: o objetivo é deixar os dados prontos. O treino começa no Lab 01, já com estes dados limpos.

| Etapa | O que fazer |
|---|---|
| 1. Inspecionar | tipos, estatísticas, faltantes e categorias |
| 2. Limpar | duplicados, categorias, outliers, alvo faltante |
| 3. Olhar | histogramas, dispersão e correlação |
| 4. Separar | features × alvo, one-hot e treino/teste |

Procure as células marcadas com **`# TODO`** e substitua os `___` pelo código certo.
Rode as células em ordem (`Shift + Enter`).

### Cola rápida de pandas
| Para… | use |
|---|---|
| ler um CSV | `pd.read_csv('arquivo.csv')` |
| tipos e faltantes de cada coluna | `df.info()` |
| estatísticas das colunas numéricas | `df.describe()` |
| faltantes por coluna | `df.isna().sum()` |
| contar valores de uma coluna | `df['col'].value_counts()` |
| remover duplicados | `df.drop_duplicates()` |
| limpar texto | `df['col'].str.strip().str.lower()` |
| remover linhas com faltante numa coluna | `df.dropna(subset=['col'])` |
| mapa de correlação | `sns.heatmap(df.corr(numeric_only=True), annot=True)` |

### Bibliotecas
No **Google Colab** tudo já vem instalado: esta célula só confere. Em outro ambiente (Jupyter no seu PC), ela instala o que faltar.

In [ ]:
import importlib.util, subprocess, sys
pacotes = {'numpy': 'numpy', 'pandas': 'pandas', 'matplotlib': 'matplotlib', 'seaborn': 'seaborn', 'sklearn': 'scikit-learn'}
faltando = [pip for modulo, pip in pacotes.items() if importlib.util.find_spec(modulo) is None]
if faltando:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *faltando])
print('Bibliotecas prontas!' if not faltando else f'Instalado agora: {", ".join(faltando)}')

## 0 · Preparação

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
print('Tudo pronto!')

### O dataset: desempenho de estudantes
Os dados abaixo são **sintéticos** (gerados aqui mesmo, para não depender de download), mas imitam problemas reais:
alguém digitou errado, algumas linhas foram copiadas duas vezes, o turno foi escrito de jeitos diferentes…

Rode a célula para criar o arquivo `alunos.csv`. Você **não precisa** entender o código de geração.

In [ ]:
def gerar_dados(n=400, seed=42):
    rng = np.random.default_rng(seed)
    horas = np.clip(rng.normal(6, 2.2, n), 0.5, 14).round(1)
    faltas = rng.poisson(3, n)
    turno = rng.choice(['manhã', 'tarde', 'noite'], n, p=[.45, .3, .25])
    monitoria = rng.choice(['sim', 'não'], n, p=[.35, .65])
    idade = rng.integers(17, 31, n)
    renda = np.round(rng.lognormal(8.2, 0.5, n), -1)          # R$ por mês, escala grande
    nota = (2.0 + 0.45 * horas - 0.25 * faltas + 0.6 * (monitoria == 'sim')
            - 0.4 * (turno == 'noite') + rng.normal(0, 0.9, n))
    nota = np.clip(nota, 0, 10).round(1)
    df = pd.DataFrame({'horas_estudo': horas, 'faltas': faltas, 'turno': turno,
                       'monitoria': monitoria, 'idade': idade, 'renda_familiar': renda,
                       'nota_final': nota})
    df['situacao'] = np.where(df['nota_final'] >= 6, 'aprovado', 'reprovado')

    # ---- problemas do mundo real ----
    sujo = rng.choice(n, 40, replace=False)
    df.loc[sujo[:14], 'turno'] = rng.choice(['Manhã', 'MANHÃ ', 'manha', 'Noite', ' tarde'], 14)
    df.loc[rng.choice(n, 20, replace=False), 'horas_estudo'] = np.nan
    df.loc[rng.choice(n, 15, replace=False), 'faltas'] = np.nan
    df.loc[rng.choice(n, 8, replace=False), 'nota_final'] = np.nan
    outl = rng.choice(n, 4, replace=False)
    df.loc[outl, 'horas_estudo'] = df.loc[outl, 'horas_estudo'].fillna(7) * 10   # vírgula esquecida
    df = pd.concat([df, df.sample(12, random_state=seed)], ignore_index=True)   # duplicados
    return df.sample(frac=1, random_state=seed).reset_index(drop=True)

gerar_dados().to_csv('alunos.csv', index=False)
print('alunos.csv criado!')

## 1 · Inspecionar

In [ ]:
# TODO: leia o arquivo 'alunos.csv' com o pandas
df = ___
df.head()

In [ ]:
# TODO: tipos de cada coluna e quantos valores não nulos ela tem
df.___()

In [ ]:
# TODO: estatísticas das colunas numéricas. Repare no máximo de horas_estudo!
df.___()

In [ ]:
# TODO: quantos valores faltantes há em cada coluna?
df.___().sum()

In [ ]:
# TODO: quais valores aparecem na coluna 'turno'?
df['turno'].___()

**Para pensar:** liste os problemas que você encontrou. Quais colunas têm faltantes? O que há de estranho em `turno` e em `horas_estudo`?

*Sua resposta:* …

## 2 · Limpar

In [ ]:
# TODO: conte as linhas duplicadas e remova-as
print('duplicados:', df.duplicated().sum())
df = df.___()
print('linhas agora:', len(df))

Para o computador, `"manhã"`, `"Manhã"` e `"MANHÃ "` são três valores diferentes.
Padronize: tire os espaços das pontas, deixe tudo minúsculo e corrija `"manha"` sem acento.

In [ ]:
# TODO: padronize a coluna turno
df['turno'] = df['turno'].str.___().str.___()
df['turno'] = df['turno'].replace({'manha': 'manhã'})
df['turno'].value_counts()

Ninguém estuda 70 horas *extras* por semana: provavelmente alguém esqueceu a vírgula (7,0 → 70).
Como **não temos certeza** do valor certo, vamos apagar esses valores (virar `NaN`).

In [ ]:
# TODO: transforme em NaN os valores de horas_estudo acima de 24
df.loc[df['horas_estudo'] > ___, 'horas_estudo'] = np.nan
df['horas_estudo'].max()

Uma linha sem `nota_final` não ensina nada: remova-a.
Os faltantes nas **features** (horas, faltas) **ficam**: no Lab 01, eles serão preenchidos usando só o treino.

In [ ]:
# TODO: remova as linhas sem nota_final
df = df.dropna(subset=[___])
print(len(df), 'linhas')
df.isna().sum()

## 3 · Olhar antes de modelar

In [ ]:
# TODO: histogramas de todas as colunas numéricas
df.___(bins=20, figsize=(12, 6))
plt.tight_layout(); plt.show()

In [ ]:
# TODO: gráfico de dispersão horas_estudo × nota_final
plt.scatter(df[___], df[___], alpha=.5)
plt.xlabel('horas de estudo'); plt.ylabel('nota final'); plt.show()

In [ ]:
# TODO: correlação entre as colunas numéricas, desenhada como um mapa de calor
corr = df.___(numeric_only=True).round(2)
sns.heatmap(corr, annot=True, cmap='coolwarm', vmin=-1, vmax=1)
plt.show()

**Para pensar:** quais features parecem ajudar a prever a nota? Alguma parece não ter relação nenhuma?
Lembre: correlação não é causalidade.

*Sua resposta:* …

## 4 · Separar
### 4.1 Features × alvo — e a coluna que “conta a resposta”
O alvo é `nota_final`. A coluna `situacao` (aprovado/reprovado) é **calculada a partir da nota**: ela só existe *depois* que a nota sai.
Usá-la como feature seria **vazamento de dados** — o modelo “colaria” a resposta.

In [ ]:
alvo = 'nota_final'
# TODO: as features são todas as colunas, menos o alvo e a coluna que vaza
X = df.drop(columns=[alvo, ___])
y = df[alvo]
X.head()

### 4.2 Categorias viram números (one-hot)
Modelos fazem contas: `turno` e `monitoria` precisam virar colunas de 0 e 1.

In [ ]:
# TODO: uma coluna 0/1 para cada categoria de turno e de monitoria
X = pd.___(X, columns=['turno', 'monitoria'], dtype=int)
X.head()

### 4.3 Separar treino e teste
O teste é a **prova inédita**: fica guardado até o fim. 80% para treino, 20% para teste.

In [ ]:
# TODO: 80% treino, 20% teste, random_state=42
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=___, random_state=42)
print(X_tr.shape, X_te.shape)

### 4.4 Salvar os dados limpos
O Lab 01 começa exatamente daqui.

In [ ]:
df.to_csv('alunos_limpo.csv', index=False)
print(df.shape, '→ alunos_limpo.csv')

## Para discutir
1. Apagar os outliers (`NaN`) ou dividir por 10: qual decisão você tomaria? O que muda nos dados?
2. Que outra coluna, num problema real, poderia “contar a resposta” como a `situacao`?
3. Por que o teste precisa ficar guardado até o fim?